[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/66_ddim_step_solution.ipynb)

# 🟡 Solution: DDIM Sampling Step

Reference solution for a single deterministic DDIM denoising step.

Given the noisy sample $x_t$, the predicted noise $\epsilon_\theta$, and the cumulative noise levels $\bar{\alpha}_t$ and $\bar{\alpha}_{t-1}$:

$$\hat{x}_0 = \frac{x_t - \sqrt{1-\bar{\alpha}_t}\,\epsilon_\theta}{\sqrt{\bar{\alpha}_t}}$$

$$x_{t-1} = \sqrt{\bar{\alpha}_{t-1}}\,\hat{x}_0 + \sqrt{1-\bar{\alpha}_{t-1}}\,\epsilon_\theta$$

In [ ]:
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def ddim_step(x_t, noise_pred, alpha_bar_t, alpha_bar_prev):
    x0_pred = (x_t - (1 - alpha_bar_t) ** 0.5 * noise_pred) / (alpha_bar_t ** 0.5)
    noise_direction = (1 - alpha_bar_prev) ** 0.5 * noise_pred
    x_prev = (alpha_bar_prev ** 0.5) * x0_pred + noise_direction
    return x_prev

In [ ]:
torch.manual_seed(42)

# Simple linear alpha_bar schedule for demo
T = 5
alpha_bars = torch.linspace(0.99, 0.01, T + 1)  # index 0..T

x_clean = torch.tensor([1.0, -1.0, 0.5])  # target signal
noise   = torch.randn_like(x_clean)

# Start from pure noise at step T
ab_T = alpha_bars[T]
x_t  = ab_T ** 0.5 * x_clean + (1 - ab_T) ** 0.5 * noise

print(f"{'Step':>4}  {'alpha_bar_t':>12}  {'x_t (first elem)':>18}")
print("-" * 42)
for step in range(T, 0, -1):
    ab_t    = alpha_bars[step]
    ab_prev = alpha_bars[step - 1]
    # Oracle noise prediction for demo
    noise_pred = (x_t - ab_t ** 0.5 * x_clean) / (1 - ab_t) ** 0.5
    x_t = ddim_step(x_t, noise_pred, ab_t, ab_prev)
    print(f"{step:>4}  {ab_prev.item():>12.4f}  {x_t[0].item():>18.4f}")

print(f"\nFinal x vs clean: {x_t.tolist()}  vs  {x_clean.tolist()}")

In [ ]:
from torch_judge import check
check("ddim_step")